In [2]:
import torch
import torch.nn as nn
from einops import rearrange, einsum
from einops.layers.torch import Rearrange


class Attention(nn.Module):
    def __init__(
            self,
            dim,
            heads = 8,
            kv_heads = 1, # How much kv heads are there?
    ):
        super(Attention, self).__init__()

        assert kv_heads <= heads and (heads % kv_heads == 0), "GQA하시려면 KV 헤드의 갯수가 query 헤드의 수의 약수여야 합니다."

        self.heads = heads
        self.kv_heads = kv_heads

        self.dim_head = dim // heads # Query head dim
        self.dim_kv = self.dim_head * kv_heads # dimensions we need for all kv heads(before splitting each)
        self.num_grouped_queries = heads // kv_heads # How many queries attend to one kv head

        self.scale = self.dim_head ** -0.5 # for attend
        self.norm = nn.RMSNorm(dim)

        # Batch, Head, Number(Sequence length), Dimension 형태로 쭈욱 활용한다. 처음 Q,K,V 분리는 B, N, D.
        # Batch는 항상 살려두고, Head끼리 적절히 쪼개 배분해 최종 concat해 반환하는 방법으로 GQA-supported attend() 함수를 만들어야 한다.

        self.qkv_split = (dim, self.dim_kv, self.dim_kv) # q, k, v
        self.to_qkv = nn.Linear(dim, sum(self.qkv_split), bias=False) # 한번에 Q, K, V 다 구해서, qkv_split으로 쪼개야 함. head로 쪼개는건 einops Rearrange 활용.

        self.split_heads = Rearrange('b n (h d) -> b h n d', d = self.dim_head)

    def forward(self, x):
        q, k, v = self.to_qkv(x).split(self.qkv_split, dim = -1)
        q, k, v = map(self.split_heads, (q, k, v))

        q = rearrange(q, 'b (h qh) ... -> b h qh ...', qh = self.num_grouped_queries)
        print(q.shape)
        sim = einsum(q, k, 'b h qh i d, b h j d -> b h qh i j') * self.scale
        print(sim.shape)
        attn = sim.softmax(dim=-1)
        print(torch.sum(attn.squeeze()[0, 0], dim=-1))
        attn_out = einsum(attn, v, 'b h qh i j, b h j d -> b h qh i d')
        print(attn_out.shape)
        attn_out = rearrange(attn_out, 'b h qh ... -> b (h qh) ...')
        print(attn_out.shape)

        return attn_out


In [3]:
from pydantic import BaseModel

class LlamaConfig(BaseModel):
    vocab_size: int
    hidden_size: int
    intermediate_ffn_dim: int
    num_hidden_layers: int
    attention_heads: int
    kv_heads: int
    ffn_activation: str # Enum this out 'gelu' 'swiglu' 'relu'
    sequence_length: int


In [4]:
class SwiGLUFFN(nn.Module):
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
    ):
        super(SwiGLUFFN, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim

        self.linear = nn.Linear(input_dim, output_dim, bias=True)
        self.swish_linear = nn.Linear(input_dim, output_dim, bias=True)

        self.downproj = nn.Linear(output_dim, input_dim, bias=True)

    def forward(self, x: torch.Tensor):
        fc_up = self.linear(x)
        fc_gate = self.swish_linear(x)

        fc_gate = nn.SiLU(fc_gate)
        fc_gate = fc_gate * fc_up

        return self.downproj(fc_gate)





In [5]:
class Llama2DecoderBlock(nn.Module):
    def __init__(
        self,
        input_dim: int,
        intermediate_dim: int,
        heads: int,
        kv_heads: int,
        activation: str,
    ):
        super(Llama2DecoderBlock, self).__init__()
        self.pre_norm = nn.RMSNorm(input_dim)
        self.attn = Attention(dim=input_dim, heads=heads, kv_heads=kv_heads)

        if activation.lower() == 'swiglu':
            self.FFN = SwiGLUFFN(input_dim, intermediate_dim)
        else:
            self.FFN = nn.Sequential(
                nn.Linear(input_dim, intermediate_dim),
                nn.GELU(),
                nn.Linear(intermediate_dim, input_dim)
            )

    def forward(self, x: torch.Tensor):
        intermediate = self.pre_norm(x)
        intermediate = self.attn(intermediate)

        intermediate += x

        intermediate = self.FFN(intermediate)

        return intermediate + x


In [6]:
class Llama2Model(nn.Module):
    def __init__(self, config: LlamaConfig):
        super(Llama2Model, self).__init__()

        self.config = config
        self.vocab_size = config.vocab_size


        self.n_layers = config.num_hidden_layers
        self.hidden_size = config.hidden_size
        self.intermediate_ffn_dim = config.intermediate_ffn_dim

        self.embedding = nn.Embedding(num_embeddings=self.vocab_size, embedding_dim=self.hidden_size)
        self.layers = nn.ModuleList()
        for _ in range(self.n_layers):
            self.layers.append(Llama2DecoderBlock(
                input_dim=self.hidden_size,
                intermediate_dim=self.hidden_size,
                heads = config.attention_heads,
                kv_heads=config.kv_heads,
                activation=config.ffn_activation,
            ))

        self.last_norm = nn.RMSNorm(self.hidden_size)
        self.w_out = nn.Linear(self.hidden_size, self.vocab_size, bias=False)

    def forward(self, x):
        h = self.embedding(x)

        for layer in self.layers:
            h = layer(h)
        h = self.last_norm(h)

        return self.w_out(h)



In [8]:
config = LlamaConfig(
    vocab_size=20000,
    hidden_size=512,
    intermediate_ffn_dim=1024,
    num_hidden_layers=8,
    attention_heads=16,
    kv_heads=4,
    ffn_activation='swiglu',
    sequence_length=100,
)

batch_size = 2
seq_length = 100
input = torch.randint(0, config.vocab_size, (batch_size, seq_length), dtype=torch.long)
model = Llama2Model(config)

output = model(input)

print(output.shape)

torch.Size([2, 4, 4, 100, 32])
torch.Size([2, 4, 4, 100, 100])
tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000],
        [1.0000, 1.0000, 1.0

RuntimeError: The size of tensor a (32) must match the size of tensor b (512) at non-singleton dimension 3